In [ ]:
import wandb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D
api = wandb.Api()

sns.set_theme(style="whitegrid")

#### Get runs

In [ ]:
runs_cpu = api.runs("mayur1009/ytm-benchmark", filters = {"group": "cifar10", "config.device": "cpu"}, lazy=False)
runs_cuda = api.runs("mayur1009/ytm-benchmark", filters = {"group": "cifar10", "config.device": "cuda"}, lazy=False)

def get_histories_with_config(runs):
    df = runs.histories(x_axis="epoch", format="pandas")
    config_map = {
        run.id: {
            "bins": run.config["bins"], 
            "approach": run.config["approach"], 
        } 
        for run in runs
    }
    df["bins"] = df["run_id"].map(lambda rid: config_map[rid]["bins"])
    df["approach"] = df["run_id"].map(lambda rid: config_map[rid]["approach"])
    return df

df_cpu = get_histories_with_config(runs_cpu)
df_cuda = get_histories_with_config(runs_cuda)

## Accuracy

In [ ]:
bins_values = sorted(df_cpu["bins"].unique())
fig, axes = plt.subplots(2, len(bins_values), figsize=(16, 8), sharey=True, layout="compressed")

for i, b in enumerate(bins_values):
    # CPU row
    cpu_b = df_cpu[df_cpu["bins"] == b]
    sns.lineplot(data=cpu_b, x="epoch", y="test_acc", hue="approach", ax=axes[0, i], marker="o", linestyle="-")
    sns.lineplot(data=cpu_b, x="epoch", y="train_acc", hue="approach", ax=axes[0, i], marker="^", linestyle="--", legend=False)
    axes[0, i].set_title(f"Bins={b}")
    axes[0, i].set_xlabel("Epoch")
    axes[0, i].get_legend().remove()
    if i == 0:
        axes[0, i].set_ylabel("CPU\nAccuracy")
    else:
        axes[0, i].set_ylabel("")

    # CUDA row
    cuda_b = df_cuda[df_cuda["bins"] == b]
    sns.lineplot(data=cuda_b, x="epoch", y="test_acc", hue="approach", ax=axes[1, i], marker="o", linestyle="-")
    sns.lineplot(data=cuda_b, x="epoch", y="train_acc", hue="approach", ax=axes[1, i], marker="^", linestyle="--", legend=False)
    axes[1, i].set_xlabel("Epoch")
    axes[1, i].get_legend().remove()
    if i == 0:
        axes[1, i].set_ylabel("CUDA\nAccuracy")
    else:
        axes[1, i].set_ylabel("")

handles, labels = axes[0, 0].get_legend_handles_labels()
handles.append(Line2D([0], [0], color="grey", linestyle="-", marker="o", label="Test"))
handles.append(Line2D([0], [0], color="grey", linestyle="--", marker="^", label="Train"))
labels.extend(["Test", "Train"])
fig.legend(handles, labels, bbox_to_anchor=(1.01, 0.5), loc="center left")
fig.suptitle("Accuracy")

## Fit Time

In [ ]:
bins_values = sorted(df_cpu["bins"].unique())
fig, axes = plt.subplots(2, len(bins_values), figsize=(16, 8), sharey="row", sharex=True, layout="compressed")

for i, b in enumerate(bins_values):
    cpu_b = df_cpu[df_cpu["bins"] == b]
    sns.lineplot(data=cpu_b[cpu_b["approach"] == "standard"], x="epoch", y="fit/time", ax=axes[0, i], label="Standard", marker="o")
    sns.lineplot(data=cpu_b[cpu_b["approach"] == "standard"], x="epoch", y="fit/time_with_encode", ax=axes[0, i], label="Standard + encode", marker="^")
    sns.lineplot(data=cpu_b[cpu_b["approach"] == "discrete"], x="epoch", y="fit/time", ax=axes[0, i], label="Discrete", marker="s")
    axes[0, i].set_title(f"Bins={b}")
    axes[0, i].set_xlabel("Epoch")
    axes[0, i].get_legend().remove()
    if i == 0:
        axes[0, i].set_ylabel("CPU\nTime (s)")
    else:
        axes[0, i].set_ylabel("")

    cuda_b = df_cuda[df_cuda["bins"] == b]
    sns.lineplot(data=cuda_b[cuda_b["approach"] == "standard"], x="epoch", y="fit/time", ax=axes[1, i], label="Standard", marker="o")
    sns.lineplot(data=cuda_b[cuda_b["approach"] == "standard"], x="epoch", y="fit/time_with_encode", ax=axes[1, i], label="Standard + encode", marker="^")
    sns.lineplot(data=cuda_b[cuda_b["approach"] == "discrete"], x="epoch", y="fit/time", ax=axes[1, i], label="Discrete", marker="s")
    axes[1, i].set_xlabel("Epoch")
    axes[1, i].get_legend().remove()
    if i == 0:
        axes[1, i].set_ylabel("CUDA\nTime (s)")
    else:
        axes[1, i].set_ylabel("")

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, bbox_to_anchor=(1.01, 0.5), loc="center left")
fig.suptitle("Fit Time")

## Inference Time

In [ ]:
bins_values = sorted(df_cpu["bins"].unique())
fig, axes = plt.subplots(2, len(bins_values), figsize=(16, 8), sharey="row", sharex=True, layout="compressed")

for i, b in enumerate(bins_values):
    cpu_b = df_cpu[df_cpu["bins"] == b]
    sns.lineplot(data=cpu_b[cpu_b["approach"] == "standard"], x="epoch", y="predict_test/time", ax=axes[0, i], label="Standard", marker="o")
    sns.lineplot(data=cpu_b[cpu_b["approach"] == "discrete"], x="epoch", y="predict_test/time", ax=axes[0, i], label="Discrete", marker="s")
    axes[0, i].set_title(f"Bins={b}")
    axes[0, i].set_xlabel("Epoch")
    axes[0, i].get_legend().remove()
    if i == 0:
        axes[0, i].set_ylabel("CPU\nTime (s)")
    else:
        axes[0, i].set_ylabel("")

    cuda_b = df_cuda[df_cuda["bins"] == b]
    sns.lineplot(data=cuda_b[cuda_b["approach"] == "standard"], x="epoch", y="predict_test/time", ax=axes[1, i], label="Standard", marker="o")
    sns.lineplot(data=cuda_b[cuda_b["approach"] == "discrete"], x="epoch", y="predict_test/time", ax=axes[1, i], label="Discrete", marker="s")
    axes[1, i].set_xlabel("Epoch")
    axes[1, i].get_legend().remove()
    if i == 0:
        axes[1, i].set_ylabel("CUDA\nTime (s)")
    else:
        axes[1, i].set_ylabel("")

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, bbox_to_anchor=(1.01, 0.5), loc="center left")
fig.suptitle("Inference Time (Test)")

## RAM Peak (Fit)

In [ ]:
bins_values = sorted(df_cpu["bins"].unique())
df_cpu["fit/ram_peak_gb"] = df_cpu["fit/ram_peak"] / 1e9
df_cuda["fit/ram_peak_gb"] = df_cuda["fit/ram_peak"] / 1e9

fig, axes = plt.subplots(2, len(bins_values), figsize=(16, 8), sharey="row", sharex=True, layout="compressed")

for i, b in enumerate(bins_values):
    cpu_b = df_cpu[df_cpu["bins"] == b]
    sns.lineplot(data=cpu_b[cpu_b["approach"] == "standard"], x="epoch", y="fit/ram_peak_gb", ax=axes[0, i], label="Standard", marker="o")
    sns.lineplot(data=cpu_b[cpu_b["approach"] == "discrete"], x="epoch", y="fit/ram_peak_gb", ax=axes[0, i], label="Discrete", marker="s")
    axes[0, i].set_title(f"Bins={b}")
    axes[0, i].set_xlabel("Epoch")
    axes[0, i].get_legend().remove()
    if i == 0:
        axes[0, i].set_ylabel("CPU\nRAM Peak (GB)")
    else:
        axes[0, i].set_ylabel("")

    cuda_b = df_cuda[df_cuda["bins"] == b]
    sns.lineplot(data=cuda_b[cuda_b["approach"] == "standard"], x="epoch", y="fit/ram_peak_gb", ax=axes[1, i], label="Standard", marker="o")
    sns.lineplot(data=cuda_b[cuda_b["approach"] == "discrete"], x="epoch", y="fit/ram_peak_gb", ax=axes[1, i], label="Discrete", marker="s")
    axes[1, i].set_xlabel("Epoch")
    axes[1, i].get_legend().remove()
    if i == 0:
        axes[1, i].set_ylabel("CUDA\nRAM Peak (GB)")
    else:
        axes[1, i].set_ylabel("")

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, bbox_to_anchor=(1.01, 0.5), loc="center left")
fig.suptitle("RAM Peak (Fit)")

## VRAM Peak (Fit) — CUDA only

In [ ]:
bins_values = sorted(df_cuda["bins"].unique())
df_cuda["fit/vram_peak_gb"] = df_cuda["fit/vram_peak"] / 1e9

fig, axes = plt.subplots(1, len(bins_values), figsize=(16, 4), sharey=True, layout="compressed")

for i, b in enumerate(bins_values):
    cuda_b = df_cuda[df_cuda["bins"] == b]
    sns.lineplot(data=cuda_b[cuda_b["approach"] == "standard"], x="epoch", y="fit/vram_peak_gb", ax=axes[i], label="Standard", marker="o")
    sns.lineplot(data=cuda_b[cuda_b["approach"] == "discrete"], x="epoch", y="fit/vram_peak_gb", ax=axes[i], label="Discrete", marker="s")
    axes[i].set_title(f"Bins={b}")
    axes[i].set_xlabel("Epoch")
    axes[i].get_legend().remove()
    if i == 0:
        axes[i].set_ylabel("VRAM Peak (GB)")
    else:
        axes[i].set_ylabel("")

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, bbox_to_anchor=(1.01, 0.5), loc="center left")
fig.suptitle("VRAM Peak (Fit) — CUDA")